In [11]:
# mean-variance portfolio allocation
import pandas as pd
import numpy as np
import math
import cvxpy as cp

from torch.autograd.functional import jacobian
from torch.autograd.functional import hessian
import torch
from scipy.stats import chi2, norm, multivariate_normal
from sklearn.neighbors import KernelDensity


class mv_portfolio:
    def __init__(self, return_mean, return_cov, lambda_1, lambda_2):
        self.return_mean = return_mean
        self.return_cov = return_cov
        self.port_num = self.return_cov.shape[0]
        self.lambda_1 = lambda_1
        self.lambda_2 = lambda_2
        # the C in the \eps = C / n
        self.ambiguity_coef = 3
        self.true_sol()
    
    def sol_obj(self, sol):
        return np.dot(np.dot(return_cov, sol), sol) + lambda_2 * np.dot(sol, sol) - lambda_1 * np.dot(return_mean, sol)
    
    def generate(self, sample_num):
        sample_data = np.random.multivariate_normal(self.return_mean, self.return_cov, size = sample_num)
        return sample_data
    def true_sol(self):
        result_array = 2 * (self.return_cov + self.lambda_2 * np.identity(self.return_cov.shape[0]))
        self.true_sol_val = np.dot(np.linalg.inv(result_array), self.lambda_1 * self.return_mean)

    def normal_idpd_pf(self, sample_data):
        mu = np.mean(sample_data, axis = 0)
        sigma2 = np.diagonal(np.cov(sample_data.T))
        return mu, sigma2



    def normal_idpd_opt(self, sample_data):
        mu, sigma2 = self.normal_idpd_pf(sample_data)

        sample_num, __ = sample_data.shape
        
        sample_data = np.random.multivariate_normal(mu, np.diag(sigma2), 100 * sample_num)
        pf_sol = self.saa_sol(sample_data)

        return pf_sol, mu, sigma2

    def normal_pf_eval(self, sol, sample_data):
        mu, sigma2 = self.normal_idpd_pf(sample_data)

        sample_num, __ = sample_data.shape
        mc_data = np.random.multivariate_normal(mu, np.diag(sigma2), 100 * sample_num)
        return self.sample_eval(sol, mc_data)

    def normal_idpd_pf_correction(self, sol, sample_data, type = 'empirical'):
        ## empirical: base estimator is the empirical estimator
        ## parametric: base estimator is the estimator under the parametric distribution

        mu, sigma2 = self.normal_idpd_pf(sample_data)
        sample_num, __ = sample_data.shape
        # compute the covariance matrix of the asymptotic parametric distribution
        ## asymptotic variance of empirical variance estimator
        Var_p = list(sigma2) + list([np.mean([(sample_data[j][i] - mu[i])** 4 for j in range(sample_num)]) - sigma2[i]**2 for i in range(self.port_num)])

        Phi_p = torch.diag(torch.from_numpy(np.array(Var_p)))
        
        x_input = torch.from_numpy(sol)
        mu_input = torch.from_numpy(mu)
        sigma2_input = torch.from_numpy(sigma2)
        dimx = self.port_num

        jacob_x_p = []
        hes_x_p = []
        for index in range(dimx):
            def func_1(params):
                mu = params[0:dimx]
                sigma2 = params[dimx:]
                delta_mu = mu - torch.from_numpy(self.return_mean)
                cov_decomp = torch.diag(sigma2) + torch.mm(delta_mu.reshape(-1, 1), delta_mu.reshape(1, -1)) + torch.from_numpy(self.lambda_2 * np.identity(self.port_num))
                return self.lambda_1 * torch.linalg.solve(cov_decomp, mu)[index] / 2
            
            params = torch.cat((mu_input, sigma2_input))
            hes = hessian(func_1, params)
            jacob = jacobian(func_1, params)

            jacob_x_p.append(jacob)
            hes_x_p.append(hes)

        ## compute hes_h_p
        if type == 'parametric':
            loop_num = 5 * sample_num
        else:
            loop_num = sample_num
        gradient_interact = 0
        sample_hes_h_p = 0

        for i in range(loop_num):
            if type == 'parametric':
                rv = torch.normal(mean = mu_input, std = torch.sqrt(sigma2_input))
            elif type == 'empirical':
                rv = torch.from_numpy(sample_data[i % sample_num])
            else:
                raise NotImplementedError
            delta_rv = rv - torch.from_numpy(self.return_mean)
            COV = torch.mm(delta_rv.reshape(-1, 1), delta_rv.reshape(1, -1))
            def cost_func(x):
                return  torch.inner(x, torch.inner(COV, x)) - self.lambda_1 * torch.inner(rv, x)  + self.lambda_2 * torch.inner(x, x)
            hes_h_x = hessian(cost_func, x_input)
            jacob_h_x = jacobian(cost_func, x_input)

            IF = list([rv[j] - mu[j] for j in range(self.port_num)]) + list([(rv[j] - mu[j])**2 - sigma2[j] for j in range(self.port_num)])
            IF = torch.from_numpy(np.array(IF))
            sample_jacob_h_p = 0
            for j in range(dimx):
                sample_jacob_h_p += jacob_h_x[j] * jacob_x_p[j]
        # used for empirical debiasing
            gradient_interact += torch.inner(sample_jacob_h_p, IF) / (loop_num ** 2)


            for j in range(dimx):
                sample_hes_h_p += (hes_h_x[j][j] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[j].reshape(1, -1)) + jacob_h_x[j] * hes_x_p[j])
                for k in range(j, dimx):
                    sample_hes_h_p += 2 * hes_h_x[j][k] * torch.mm(jacob_x_p[j].reshape(-1, 1), jacob_x_p[k].reshape(1, -1))

        I_h_p = sample_hes_h_p/loop_num
        # here the esimators we apply are all unbiased, so there is no higher-order bias...

        
        if type == 'parametric':
            return torch.trace(torch.mm(I_h_p, Phi_p)).numpy() / (2 * sample_num)
        elif type == 'empirical':
            return -gradient_interact.numpy()
            
    def chi2_DRO_sol(self, sample_data):
        sample_size, __ = sample_data.shape
        weight = cp.Variable(self.port_num)
        eta = cp.Variable()
        aux_s = cp.Variable(sample_size)

        loss = math.sqrt((1 + self.ambiguity_coef / sample_size) /sample_size) * cp.norm(cp.pos(aux_s - eta), 2) + eta

        cons = []
        for i in range(sample_size):
            product_term = sample_data[i] - self.return_mean
            cons += [aux_s[i] >= cp.quad_form(weight, np.dot(product_term.reshape(-1, 1), product_term.reshape(1, -1))) - self.lambda_1 * weight @ sample_data[i] + self.lambda_2 * cp.norm(weight, 2) ** 2] 
        
        prob = cp.Problem(cp.Minimize(loss), cons)
        prob.solve(solver = cp.GUROBI)

        return weight.value

    def chi2dro_sol_correction(self, sol, sample_data):
        # sample_num, __ = sample_data.shape
        # psi_0 = 0.5
        # diff = sample_data - self.return_mean
        # diff_compute = np.zeros((sample_data.shape[0], self.port_num, self.port_num))
        # for i in range(sample_data.shape[0]):
        #     diff_compute[i] = np.dot(diff[i].reshape(-1, 1), diff[i].reshape(1, -1)) + self.lambda_2 * np.identity(self.port_num)
        # I_h_inv = np.linalg.inv(np.mean(diff_compute, axis = 0)) / 2
        
        # J_h = np.zeros((sample_data.shape[0], self.port_num, self.port_num))
        # grad = np.zeros((sample_num, self.port_num))
        # val = np.zeros(sample_num)
        # for i in range(sample_data.shape[0]):
        #     product_term = 2 * np.dot(diff_compute[i], sol) - self.lambda_1 * sample_data[i]
        #     grad[i] = product_term
        #     val[i] = self.sample_eval(sol, sample_data[i:(i + 1)])
        # grad_val_interact = [np.cov(val, grad.T[i])[0][1] for i in range(self.port_num)]
        # return self.saa_correction(sol, sample_data) - math.sqrt(self.ambiguity_coef / (2 * sample_num * np.var(val))) * np.dot(I_h_inv, grad_val_interact).dot(np.mean(grad, axis = 0))
        return self.saa_correction(sol, sample_data)
    
    def saa_sol(self, sample_data):
        sample_mean = np.mean(sample_data, axis = 0)
        cov_info = np.zeros((sample_data.shape[0], sample_data.shape[1], sample_data.shape[1]))
        for i in range(sample_data.shape[0]):
            product_term = sample_data[i] - self.return_mean
            cov_info[i] = np.dot(product_term.reshape(-1, 1), product_term.reshape(1, -1))
        sample_cov = np.mean(cov_info, axis = 0) 


        return_array = 2 * (sample_cov + self.lambda_2 * np.identity(sample_cov.shape[0]))
        
        self.saa_sol_val = np.dot(np.linalg.inv(return_array), self.lambda_1 * sample_mean)
        return self.saa_sol_val
    


    def uniform_sol(self, sample_data):
        sample_mean = np.mean(sample_data, axis = 0)
        cov_info = np.zeros((sample_data.shape[0], sample_data.shape[1], sample_data.shape[1]))
        for i in range(sample_data.shape[0]):
            product_term = sample_data[i] - self.return_mean
            cov_info[i] = np.dot(product_term.reshape(-1, 1), product_term.reshape(1, -1))
        sample_cov = np.mean(cov_info, axis = 0)

        


        theta = self.lambda_1 * np.sum(sample_mean) / (2 * (np.sum(sample_cov) + self.lambda_2 * self.port_num))
        self.uniform_sol_val = theta * np.ones(self.port_num)
        return self.uniform_sol_val

    def uniform_sol_KDE(self, sample_data):
        sample_num, port_num = sample_data.shape
        h_n = 2 * sample_num ** (-1/(port_num + 4))
        kde = KernelDensity(kernel = 'gaussian', bandwidth = h_n).fit(sample_data)
        mc_num = 10000
        mc_data = kde.sample(mc_num)
        return self.uniform_sol(mc_data)


    def uniform_correction_KDE(self, sol, sample_data):
        sample_num, port_num = sample_data.shape
        assert (port_num <= 3)

        h_n = 2 * sample_num ** (-1/(port_num + 4))
        kde = KernelDensity(kernel = 'gaussian', bandwidth = h_n).fit(sample_data)
        import itertools
        lw, up = np.min(sample_data, axis = 0), np.max(sample_data, axis = 0)
        #create sample grid for finite-integral simulation
        mc_int_dim_num = 100
        mc_int_num = mc_int_dim_num ** port_num
        

        if port_num == 1:
            rv = np.arange(start = lw[0], stop = up[0], step = (up[0] - lw[0]) / mc_int_dim_num)
        elif port_num == 2:
            rv0 = np.arange(start = lw[0], stop = up[0], step = (up[0] - lw[0]) / mc_int_dim_num)
            rv1 = np.arange(start = lw[1], stop = up[1], step = (up[1] - lw[1]) / mc_int_dim_num)
            rv = list([np.array(item) for item in itertools.product(rv0, rv1)])
        else:
            rv0 = np.arange(start = lw[0], stop = up[0], step = (up[0] - lw[0]) / mc_int_dim_num)
            rv1 = np.arange(start = lw[1], stop = up[1], step = (up[1] - lw[1]) / mc_int_dim_num)
            rv2 = np.arange(start = lw[2], stop = up[2], step = (up[2] - lw[2]) / mc_int_dim_num)
            rv = list([np.array(item) for item in itertools.product(rv0, rv1, rv2)])

        gradient_interact = 0
        for i in range(sample_num):
            int_val = 0
            for k in range(mc_int_num):
                var = multivariate_normal(mean = sample_data[i], cov = h_n * np.identity(port_num))
                rv_matrix = np.dot((rv[k] - self.return_mean).reshape(-1, 1), (rv[k] - self.return_mean).reshape(1, -1))
                int_val -= (var.pdf(rv[k]) - math.exp(kde.score_samples(rv[k].reshape(1, -1)))) * \
                            ((np.sum(rv_matrix) + self.lambda_2 * port_num) * 2 * sol[0] - self.lambda_1 * np.sum(rv[k])) / mc_int_num

            rv_matrix = np.dot((sample_data[i] - self.return_mean).reshape(-1, 1), (sample_data[i] - self.return_mean).reshape(1, -1))
            gradient = (np.sum(rv_matrix) + self.lambda_2 * port_num) * 2 * sol[0] - self.lambda_1 * np.sum(sample_data[i])
            gradient_interact += gradient * int_val

        hessian_val = 0
        for k in range(mc_int_num):
            rv_matrix = np.dot((rv[k] - self.return_mean).reshape(-1, 1), (rv[k] - self.return_mean).reshape(1, -1))
            hessian_val += 2 * (np.sum(rv_matrix) + self.lambda_2 * port_num)/ mc_int_num

        return gradient_interact/ (hessian_val * sample_num ** 2)





    def sample_eval(self, sol, sample_data):
        sample_mean = np.mean(sample_data, axis = 0)
        cov_info = np.zeros((sample_data.shape[0], sample_data.shape[1], sample_data.shape[1]))
        for i in range(sample_data.shape[0]):
            product_term = sample_data[i] - self.return_mean
            cov_info[i] = np.dot(product_term.reshape(-1, 1), product_term.reshape(1, -1))
        sample_cov = np.mean(cov_info, axis = 0)

        return np.dot(sample_cov, sol).dot(sol) - self.lambda_1 * np.dot(sample_mean, sol) + self.lambda_2 * np.dot(sol, sol)



    def true_eval(self, sol):
        return np.dot(self.return_cov, sol).dot(sol) - self.lambda_1 * np.dot(self.return_mean, sol) + self.lambda_2 * np.dot(sol, sol)



    def uniform_correction(self, sol, sample_data):
        #compute the term: \sum_{i,j}^d A + \lambda_2 d
        #sample_cov = np.cov(sample_data.T)
        sample_mean = np.mean(sample_data, axis = 0)
        diff = sample_data - self.return_mean
        diff_compute = np.zeros(sample_data.shape[0])
        for i in range(sample_data.shape[0]):
            diff_compute[i] = np.sum(np.dot(diff[i].reshape(-1, 1), diff[i].reshape(1, -1))) + self.lambda_2 * self.port_num
        I_h_inv = 1/np.mean(2 * diff_compute)
        J_h = np.zeros(sample_data.shape[0])
        for i in range(sample_data.shape[0]):
            J_h[i] = (2 * sol[0] * diff_compute[i] - self.lambda_1 * np.sum(sample_data[i])) ** 2
        return I_h_inv * np.mean(J_h) / (sample_data.shape[0])
        #return I_h_inv, np.mean(J_h)


    # we use torch to store the data
    def IO_correction(self, I_h, J_h, sample_num):
        bias_correction = torch.trace(torch.linalg.solve(I_h, J_h))
        return bias_correction.numpy() / sample_num



    def uniform_correction_torch(self, sol, sample_data):
        sample_num, __ = sample_data.shape
        I_h, J_h  = 0, 0
        x_input = torch.from_numpy(sol[0:1])
        for i in range(sample_num):
            def cost_func(x):
                diff = sample_data[i] - self.return_mean

                return (np.sum(np.dot(diff.reshape(-1, 1), diff.reshape(1, -1))) + self.lambda_2 * self.port_num) * torch.square(x) \
                        - self.lambda_1 * np.sum(sample_data[i]) * x 
            hes_h_x = hessian(cost_func, x_input)
            jacob_h_x = jacobian(cost_func, x_input)

            I_h += hes_h_x 
            J_h += torch.mm(jacob_h_x.reshape(-1, 1), jacob_h_x.reshape(1, -1))
        return self.IO_correction(I_h / sample_num, J_h / sample_num, sample_num)



    def uniform_cover_correction_compute(self, sol, sample_data):
        I_h_inv, J_h = self.uniform_correction(sol, sample_data)
        lw_bd, up_bd = chi2.ppf(0.005, 1), chi2.ppf(0.995, 1)
        base = self.sample_eval(sol, sample_data)
        base_correct = base + I_h_inv * J_h / (2 * sample_data.shape[0])
        cover_0, cover_1 = 0, 0
        # without correction
        if self.true_eval(sol) >= base + lw_bd * I_h_inv * J_h / sample_data.shape[0] \
            and self.true_eval(sol) <= base + up_bd * I_h_inv * J_h / sample_data.shape[0]:
                cover_0 = 1
        # with correction
        #+ lw_bd * I_h_inv * J_h / sample_data.shape[0] 
        if self.true_eval(sol) >= base_correct \
            and self.true_eval(sol) <= base_correct + 5 * up_bd * I_h_inv * J_h / sample_data.shape[0]:
                cover_1 = 1
        return cover_0, cover_1

    def saa_correction(self, sol, sample_data):
        sample_mean = np.mean(sample_data, axis = 0)
        diff = sample_data - self.return_mean
        diff_compute = np.zeros((sample_data.shape[0], self.port_num, self.port_num))
        for i in range(sample_data.shape[0]):
            diff_compute[i] = np.dot(diff[i].reshape(-1, 1), diff[i].reshape(1, -1)) + self.lambda_2 * np.identity(self.port_num)
        I_h_inv = np.linalg.inv(np.mean(diff_compute, axis = 0)) / 2
        J_h = np.zeros((sample_data.shape[0], self.port_num, self.port_num))
        for i in range(sample_data.shape[0]):
            product_term = 2 * np.dot(diff_compute[i], sol) - self.lambda_1 * sample_data[i]
            J_h[i] = np.dot(product_term.reshape(-1, 1), product_term.reshape(1, -1))
        J_h_final = np.mean(J_h, axis = 0)
        bias = np.dot(I_h_inv, J_h_final)
        #return I_h_inv, J_h_final
        return bias.trace()/ (sample_data.shape[0])

    def saa_cover_correction_compute(self, sol, sample_data):
        simulate_size = 10000
        I_h_inv, J_h = self.saa_correction(sol, sample_data)
        I_h = np.linalg.inv(I_h_inv)

        # simulation to find the upper and lower 
        cov = np.dot(I_h_inv, J_h).dot(I_h_inv) / sample_data.shape[0]
        seq = np.random.multivariate_normal(np.zeros(sample_data.shape[1]), cov, simulate_size)
        simulate_output = np.zeros(simulate_size)
        for i in range(simulate_size):
            simulate_output[i] = np.dot(seq[i], I_h).dot(seq[i])
        ## find quantile
        lw_bd = np.quantile(simulate_output, 0.005)
        up_bd = np.quantile(simulate_output, 0.995)

        base = self.sample_eval(sol, sample_data)
        base_correct = base + np.dot(I_h_inv, J_h).trace()/(2 * sample_data.shape[0])
        cover_0, cover_1 = 0, 0
        # without correction
        if self.true_eval(sol) >= base + lw_bd \
            and self.true_eval(sol) <= base + up_bd:
                cover_0 = 1
        # with correction
        if self.true_eval(sol) >= base_correct + lw_bd \
            and self.true_eval(sol) <= base_correct + up_bd:
                cover_1 = 1
        return cover_0, cover_1




loop_num = 100



saa_bias_0 = np.zeros(loop_num)
saa_bias_1 = np.zeros(loop_num)
saa_bias_true = np.zeros(loop_num)
uniform_bias_0 = np.zeros(loop_num)
uniform_bias_1 = np.zeros(loop_num)
uniform_bias_true = np.zeros(loop_num)



for sample_num in [50]:
    correct_num_1 = 0
    correct_num_0 = 0
    correct_value_1 = 0
    correct_value_0 = 0
    saa_correction_all = 0
    uniform_correction_all = 0

    cover_uniform_0 = 0
    cover_uniform_1 = 0
    cover_saa_0 = 0
    cover_saa_1 = 0
    error = np.zeros((8, loop_num))
    for k in range(loop_num):
        print(k)
        #sample_num = 20
        # base case: all choose the same weigght
        asset_num = 2
        return_mean = np.random.uniform(0, 4, asset_num)
        return_cov = np.diag(1 * np.ones(asset_num))
        for i in range(asset_num):
            for j in range(i + 1, asset_num):
                rad_val = 0.5 * np.random.rand() 
                return_cov[i][j], return_cov[j][i] = 0.5, 0.5

        lambda_1 = 1
        lambda_2 = 2
        assert (lambda_1 >=0 and lambda_2 >= 0)



        mv_model = mv_portfolio(return_mean, return_cov, lambda_1, lambda_2)
        #best_sol = best_true_sol(return_mean, return_cov, lambda_1, lambda_2)

        sample_data = mv_model.generate(sample_num)



        saa_sol = mv_model.saa_sol(sample_data)
        uniform_sol = mv_model.uniform_sol(sample_data)
        #u_KDE_sol = mv_model.uniform_sol_KDE(sample_data)
        chi2dro_sol = mv_model.chi2_DRO_sol(sample_data)

        error[0][k] = mv_model.true_eval(chi2dro_sol) - mv_model.sample_eval(chi2dro_sol, sample_data)
        error[1][k] = error[0][k] - mv_model.chi2dro_sol_correction(chi2dro_sol, sample_data)
        #error[2][k] = error[0][k] - mv_model.saa_correction(chi2dro_sol, sample_data)
        
        error[3][k] = mv_model.true_eval(saa_sol) - mv_model.sample_eval(saa_sol, sample_data)
        error[4][k] = error[3][k] - mv_model.saa_correction(saa_sol, sample_data)

        # error[2][k] = mv_model.true_eval(u_KDE_sol) - mv_model.sample_eval(u_KDE_sol, sample_data)
        # error[3][k] = error[2][k] - mv_model.uniform_correction_KDE(u_KDE_sol, sample_data)

        # # compare the result in torch computation
        # error[2][k] = mv_model.true_eval(uniform_sol) - mv_model.sample_eval(uniform_sol, sample_data)
        # error[3][k] = error[2][k] - mv_model.uniform_correction(uniform_sol, sample_data)

        # pf_idpd_sol, __, __ = mv_model.normal_idpd_opt(sample_data)
        # error[4][k] = mv_model.true_eval(pf_idpd_sol) - mv_model.normal_pf_eval(pf_idpd_sol, sample_data)
        # error[5][k] = error[4][k] - mv_model.normal_idpd_pf_correction(pf_idpd_sol, sample_data, 'parametric')


        # error[6][k] = mv_model.true_eval(pf_idpd_sol) - mv_model.sample_eval(pf_idpd_sol, sample_data)
        # error[7][k] = error[6][k] - mv_model.normal_idpd_pf_correction(pf_idpd_sol, sample_data)

        #print(error, error_debias) 
        # a, b = mv_model.uniform_cover_correction_compute(uniform_sol, sample_data)
        # cover_uniform_0 += a
        # cover_uniform_1 += b

        # a, b = mv_model.saa_cover_correction_compute(saa_sol, sample_data)
        # cover_saa_0 += a
        # cover_saa_1 += b

        # saa_debias = mv_model.saa_correction(saa_sol, sample_data) + mv_model.sample_eval(saa_sol, sample_data)
        # uniform_debias = mv_model.uniform_correction(uniform_sol, sample_data) + mv_model.sample_eval(uniform_sol, sample_data)

        #check whether it leads to a better selection procedure
        # if (saa_debias < uniform_debias and mv_model.true_eval(saa_sol) < mv_model.true_eval(uniform_sol)) or (saa_debias >= uniform_debias and mv_model.true_eval(saa_sol) >= mv_model.true_eval(uniform_sol)):
        #     correct_num_1 += 1
        
        # if (mv_model.sample_eval(saa_sol, sample_data) < mv_model.sample_eval(uniform_sol, sample_data) and mv_model.true_eval(saa_sol) < mv_model.true_eval(uniform_sol)) \
        #     or (mv_model.sample_eval(saa_sol, sample_data) >= mv_model.sample_eval(uniform_sol, sample_data) and mv_model.true_eval(saa_sol) >= mv_model.true_eval(uniform_sol)):
        #     correct_num_0 += 1
        
        # #result of model selection
        # if saa_debias < uniform_debias:
        #     correct_value_1 += mv_model.true_eval(saa_sol)
        # else:
        #     correct_value_1 += mv_model.true_eval(uniform_sol)

        # if mv_model.sample_eval(saa_sol, sample_data) < mv_model.sample_eval(uniform_sol, sample_data):
        #     correct_value_0 += mv_model.true_eval(saa_sol)
        # else:
        #     correct_value_0 += mv_model.true_eval(uniform_sol)

    #print(cover_uniform_0/loop_num, cover_uniform_1/loop_num, cover_saa_0/loop_num, cover_saa_1/loop_num)

        
#         saa_bias_0[k] = mv_model.true_eval(saa_sol) - mv_model.sample_eval(saa_sol, sample_data)
#         saa_bias_1[k] = mv_model.true_eval(saa_sol) - saa_debias
#         saa_bias_true[k] = mv_model.true_eval(saa_sol) - mv_model.saa_correction(mv_model.true_sol_val, sample_data) - mv_model.sample_eval(saa_sol, sample_data)

#         uniform_bias_0[k] = mv_model.true_eval(uniform_sol) - mv_model.sample_eval(uniform_sol, sample_data)
#         uniform_bias_1[k] = mv_model.true_eval(uniform_sol) - uniform_debias
#         uniform_bias_true[k] = mv_model.true_eval(uniform_sol) - mv_model.uniform_correction(mv_model.true_sol_val, sample_data) - mv_model.sample_eval(uniform_sol, sample_data)


# # print(np.mean(abs(saa_bias_0)), np.std(saa_bias_0), np.mean(abs(saa_bias_1)), np.std(saa_bias_1), np.mean(abs(saa_bias_true)), np.std(saa_bias_true))
# # print(np.mean(abs(uniform_bias_0)), np.std(uniform_bias_0), np.mean(abs(uniform_bias_1)), np.std(uniform_bias_1), np.mean(abs(uniform_bias_true)), np.std(uniform_bias_true))
#     #print(mv_model.saa_correction(saa_sol, sample_data), mv_model.uniform_correction(uniform_sol, sample_data))
#     #print(mv_model.sample_eval(saa_sol, sample_data), saa_debias, mv_model.sample_eval(mv_model.true_sol_val, sample_data), mv_model.true_eval(mv_model.true_sol_val))
#         saa_correction_all += mv_model.saa_correction(saa_sol, sample_data)
#         uniform_correction_all += mv_model.uniform_correction(uniform_sol, sample_data)

#     #print(sample_num, np.mean(saa_bias_0), np.mean(saa_bias_1), np.mean(uniform_bias_0), np.mean(uniform_bias_1))
#     print(sample_num, saa_correction_all / loop_num, uniform_correction_all / loop_num, saa_correction_all / uniform_correction_all)
    #print(sample_num, correct_value_0 / loop_num, correct_value_1 / loop_num)
    #print('prob of correct', correct_num_0 / loop_num)
    #print('============')
#print('uniform debias', mv_model.sample_eval(uniform_sol, sample_data), uniform_debias, )
    print(np.mean(error, axis = 1))


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
[ 0.01134914 -0.0039435   0.          0.01113176 -0.00676346  0.
  0.          0.        ]


In [27]:
from src.nv_model import newsvendor
import pandas as pd
import numpy as np
mode = 'exponential'
sample_num = 50
nv_model = newsvendor(5, 2, mode)
data = nv_model.generate(sample_num)

exp_sol = nv_model.exp(data)

true = nv_model.true_eval(exp_sol)
exp_estimate = nv_model.exp_eval(exp_sol, data) + nv_model.exp_correction_MS(exp_sol, data)

normal_sol = nv_model.normal(data)
normal_estimate = nv_model.normal_eval(normal_sol, data) + nv_model.normal_correction_MS(normal_sol, data)


print(true, normal_estimate, exp_estimate)

-115.42232270084926 -125.89237778888229 -109.9738661092685


In [23]:
a = np.array([1,2])
b = np.array([[1,4, 5],[0,2, 3]])
np.mean(b, axis = 0)

array([0.5, 3. , 4. ])

In [1]:
import torch

In [20]:
var = multivariate_normal(mean = [0, 0], cov = [[1, 0], [0, 1]])
var.pdf([0, 0])

0.15915494309189535

In [9]:
import numpy as np
import itertools
a = np.array([1,2,3])
b = np.array([4,5,6])
c = np.array([7,8,9])

for item in itertools.product(a, b, c):
    print(list(item))

[array([1, 2, 3])]
[array([4, 5, 6])]
[array([7, 8, 9])]


In [1]:
import torch

In [4]:
pf_idpd_sol

(array([0.49883383, 0.41358284]),
 array([2.80612216, 2.48386859]),
 array([0.76881946, 0.90605769]))

In [1]:
import torch

In [25]:
chi2.ppf(0.975, 1)

5.023886187314888

In [124]:
print(np.mean(saa_bias_0) ** 2 + np.std(saa_bias_0) ** 2)
print(np.mean(saa_bias_0 ** 2))


0.18365529853673468
0.18365529853673465


In [6]:
a = 1
b = 2 
b += 1

In [4]:
import numpy as np
a = np.random.multivariate_normal(np.zeros(3), np.identity(3), 5)
np.dot(a[0], np.identity(3)).dot(a[0])

1.4427192091910948

In [74]:
print(mv_model.sample_eval(saa_sol, sample_data), mv_model.sample_eval(mv_model.true_sol_val, sample_data), mv_model.true_eval(mv_model.true_sol_val))

-4.149088537598516 -4.071231443329097 -4.453362429075783


In [26]:
print(saa_sol)
print(uniform_sol)
print(mv_model.true_sol_val)


[0.23611511 0.11864554 0.48941208 0.55192484 0.06306925]
[0.44085671 0.44085671 0.44085671 0.44085671 0.44085671]
[0.24440339 0.17937801 0.51749054 0.54375133 0.11174802]


In [4]:
np.sum(np.dot(sample_mean.reshape(-1, 1), sample_mean.reshape(1, -1)))

array([[15.06848432, 11.67237908, 16.53283261,  8.72022317, 13.58353278],
       [11.67237908,  9.04168134, 12.80669545,  6.7548765 , 10.52210299],
       [16.53283261, 12.80669545, 18.13948559,  9.5676504 , 14.90357417],
       [ 8.72022317,  6.7548765 ,  9.5676504 ,  5.04644598,  7.86087272],
       [13.58353278, 10.52210299, 14.90357417,  7.86087272, 12.24491852]])

In [10]:
a = np.zeros((2,2,2))
a[0] = np.array([[1,0],[0,3]])
a[1] = np.array([[3,0],[0,6]])
np.mean(np.linalg.inv(a), axis = 0)

array([[0.66666667, 0.        ],
       [0.        , 0.25      ]])